# 🎭 ChuckleNet: Multilingual 500+ Video Pipeline

**Goal**: Build 500+ video laughter dataset with 10%+ positive rate, multilingual (en/hi/zh/es)

## Architecture
```
YouTube Search → yt-dlp download → Extract audio → VTT/caption parsing
    → Utterance segmentation → F0 extraction → [laughter] labeling → Train/Eval
```

## Data Sources
| Source | Videos | Label Method | Expected Positive |
|--------|--------|--------------|-------------------|
| YouTube Comedy (en) | 200 | Auto-captions | 15-25% |
| YouTube Comedy (hi) | 100 | Hindi captions | 15-25% |
| YouTube Comedy (zh) | 100 | Chinese captions | 15-25% |
| YouTube Comedy (es) | 100 | Spanish captions | 15-25% |

## Key Insight
YouTube auto-captions include `[laughter]` markers. Group words into utterances
(pause-gapped), label utterance positive if ANY `[laughter]` overlaps.
This gives 15-25% positive rate (same as 87-video dataset).

**Runtime**: ~3 hours with T4 GPU

In [ ]:
# 1. Setup
!pip install yt-dlp librosa numpy pandas scikit-learn tqdm 2>&1 | tail -3
!apt-get install -y ffmpeg 2>&1 | tail -2
import os, re, json, time, subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/chuckle_net_multilingual'
os.makedirs(BASE, exist_ok=True)
os.makedirs(f'{BASE}/audio', exist_ok=True)
os.makedirs(f'{BASE}/vtt', exist_ok=True)
print('Setup done, base:', BASE)

In [ ]:
# 2. YouTube Search + Download Functions
def search_youtube_comedy(query, max_results=20, lang='en'):
    """Search YouTube for comedy videos using yt-dlp."""
    search_query = f"{query} standup comedy full special lang:{lang}"
    cmd = [
        'yt-dlp', '--flat-playlist', '--print', '%(id)s||%(title)s||%(duration)s',
        f'ytsearch{max_results}:{search_query}'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    videos = []
    for line in result.stdout.strip().split('\n'):
        if '||' in line:
            parts = line.split('||')
            if len(parts) == 3:
                vid_id, title, duration = parts
                try:
                    duration = int(duration) if duration.isdigit() else 0
                    if duration >= 600:  # At least 10 min
                        videos.append({'id': vid_id, 'title': title, 'duration': duration})
                except:
                    pass
    return videos

def download_video_with_captions(video_id, output_dir, lang='en'):
    """Download video + captions using yt-dlp."""
    url = f'https://www.youtube.com/watch?v={video_id}'
    
    # Download captions (VTT)
    cmd_captions = [
        'yt-dlp', '--write-auto-sub', f'--sub-langs={lang}',
        '--skip-download', '--convert-subs=vtt',
        '-o', f'{output_dir}/%(id)s.%(ext)s', url
    ]
    
    # Download audio
    cmd_audio = [
        'yt-dlp', '-f', 'bestaudio[ext=m4a]',
        '--extract-audio', '--audio-format', 'wav',
        '--audio-quality', '5',
        '-o', f'{output_dir}/%(id)s.wav', url
    ]
    
    # Try captions first
    subprocess.run(cmd_captions, capture_output=True, timeout=60)
    
    # Then audio
    result = subprocess.run(cmd_audio, capture_output=True, text=True, timeout=300)
    
    return result.returncode == 0

# Test
test_videos = search_youtube_comedy('Russell Peters', max_results=5, lang='en')
print(f'Found {len(test_videos)} videos')
for v in test_videos[:3]:
    print(f"  {v['id']}: {v['title'][:50]} ({v['duration']}s)")

In [ ]:
# 3. Download Comedy Videos by Language
# Languages: en, hi, zh, es
LANGUAGES = {
    'en': 'Russell Peters, Dave Chappelle, Ali Wong, Kevin Hart',
    'hi': 'Zakir Khan, Vir Das, Kanan Gill, Bisno',
    'zh': 'Chinese stand up comedy, comedy special',
    'es': 'Stand up comedy español, comedia especial'
}

all_video_ids = []

for lang, queries in LANGUAGES.items():
    print(f'\n=== Searching {lang.upper()} comedy ===')
    for query in queries.split(', '):
        videos = search_youtube_comedy(query, max_results=30, lang=lang)
        print(f'  {query}: {len(videos)} videos')
        all_video_ids.extend([(v['id'], lang) for v in videos])

# Deduplicate
seen = set()
unique = []
for vid, lang in all_video_ids:
    if vid not in seen:
        seen.add(vid)
        unique.append((vid, lang))

print(f'\nTotal unique videos: {len(unique)}')
for lang in LANGUAGES:
    count = sum(1 for v, l in unique if l == lang)
    print(f'  {lang}: {count}')

In [ ]:
# 4. Batch Download (run in batches to avoid rate limiting)
def download_batch(video_ids, output_dir, batch_size=10, delay=5):
    """Download with rate limiting."""
    success, failed = 0, 0
    for i, vid in enumerate(tqdm(video_ids, desc='Downloading')):
        try:
            ok = download_video_with_captions(vid, output_dir)
            if ok:
                success += 1
            else:
                failed += 1
            
            # Rate limit
            if (i + 1) % batch_size == 0:
                time.sleep(delay)
        except Exception as e:
            failed += 1
    
    return success, failed

# Download first 200 (en) as test
en_vids = [v for v, l in unique if l == 'en'][:200]
print(f'Downloading {len(en_vids)} English videos...')
s, f = download_batch(en_vids, f'{BASE}/audio')
print(f'  Success: {s}, Failed: {f}')

In [ ]:
# 5. Parse VTT + Extract Utterance-Level F0 Features
import librosa

def parse_vtt_cues(vtt_path):
    """Parse VTT, return list of (start, end, text, has_laughter)."""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    def to_sec(ts):
        ts = ts.replace('.', ':')
        p = ts.split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    
    cues = []
    lines = content.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if '-->' in line:
            start, end = line.split('-->')
            start, end = to_sec(start.strip()), to_sec(end.strip())
            
            # Collect text lines
            text_lines = []
            i += 1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                text_lines.append(lines[i].strip())
                i += 1
            
            text = ' '.join(text_lines)
            has_laughter = '[laughter]' in text.lower()
            cues.append((start, end, text, has_laughter))
        else:
            i += 1
    
    return cues

def extract_f0_features(y, sr=22050, hop_length=512):
    """Extract 5-dim F0 features from audio segment."""
    try:
        f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, hop_length=hop_length)
        f0 = np.nan_to_num(f0, nan=0.0)
        
        return [
            float(np.mean(f0)),
            float(np.std(f0)),
            float(np.max(f0)),
            float(np.min(f0)),
            float(np.mean(f0 > 0))
        ]
    except:
        return [0, 0, 0, 0, 0]

def process_video(audio_path, vtt_path):
    """Process one video: parse VTT + extract F0."""
    cues = parse_vtt_cues(vtt_path)
    if not cues:
        return None, None
    
    y, sr = librosa.load(audio_path, sr=22050)
    
    features, labels = [], []
    for start, end, text, has_laughter in cues:
        if end <= start:
            continue
        y_seg = y[int(start*sr):int(end*sr)]
        if len(y_seg) < sr * 0.1:  # Skip < 100ms
            continue
        feat = extract_f0_features(y_seg, sr)
        features.append(feat)
        labels.append(1 if has_laughter else 0)
    
    return np.array(features), np.array(labels)

# Test on one video
import glob
test_audio = glob.glob(f'{BASE}/audio/*.wav')[0] if glob.glob(f'{BASE}/audio/*.wav') else None
test_vtt = glob.glob(f'{BASE}/vtt/*.vtt')[0] if glob.glob(f'{BASE}/vtt/*.vtt') else None

if test_audio and test_vtt:
    feats, labs = process_video(test_audio, test_vtt)
    print(f'Test: features={feats.shape}, positive={labs.sum()}/{len(labs)} ({100*labs.mean():.1f}%)')
else:
    print('No test files yet - download first')

In [ ]:
# 6. Batch Process All Videos
all_features, all_labels, all_vids, all_langs = [], [], [], []

audio_files = {os.path.splitext(os.path.basename(f))[0]: f 
               for f in glob.glob(f'{BASE}/audio/*.wav')}
vtt_files = {os.path.splitext(os.path.splitext(os.path.basename(f))[0])[0]: f 
             for f in glob.glob(f'{BASE}/vtt/*.vtt')}

common_ids = set(audio_files.keys()) & set(vtt_files.keys())
print(f'Videos with both audio+VTT: {len(common_ids)}')

for vid in tqdm(common_ids, desc='Processing'):
    try:
        feats, labs = process_video(audio_files[vid], vtt_files[vid])
        if feats is None or len(feats) == 0:
            continue
        
        # Get language
        lang = next((l for v, l in unique if v == vid), 'en')
        
        for f, l in zip(feats, labs):
            all_features.append(f)
            all_labels.append(l)
            all_vids.append(vid)
            all_langs.append(lang)
    except Exception as e:
        continue

X = np.array(all_features)
y = np.array(all_labels)
vids = np.array(all_vids)
langs = np.array(all_langs)

print(f'\nTotal: {len(X)} segments from {len(set(vids))} videos')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

# Language breakdown
for lang in LANGUAGES:
    mask = langs == lang
    if mask.sum() > 0:
        print(f'  {lang}: {mask.sum()} segs, {100*y[mask].mean():.1f}% positive')

In [ ]:
# 7. Train + Evaluate F0 Model (Video-level split)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

# Split by VIDEO (not random)
unique_vids = list(set(vids))
np.random.seed(42)
np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids) * 0.2))
test_vids = set(unique_vids[:n_test])
train_vids = set(unique_vids[n_test:])

train_mask = np.isin(vids, list(train_vids))
test_mask = ~train_mask

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% positive)')
print(f'Test:  {len(X_test)} ({100*y_test.mean():.1f}% positive)')

# Logistic Regression
print('\n=== Logistic Regression ===')
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

# MLP
print('\n=== MLP ===')
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print(f'F1: {f1_score(y_test, y_pred_mlp):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_mlp):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_mlp):.4f}')

In [ ]:
# 8. Save Dataset
out = {
    'features': X,
    'labels': y,
    'vids': vids,
    'langs': langs
}
np.savez_compressed(f'{BASE}/multilingual_500plus.npz', **out)
print(f'Saved to {BASE}/multilingual_500plus.npz')

# Save model
import pickle
with open(f'{BASE}/f0_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
print(f'Model saved to {BASE}/f0_model.pkl')

In [ ]:
# 9. Multilingual Analysis
print('=== Multilingual F1 by Language ===')
for lang in LANGUAGES:
    mask = langs == lang
    if mask.sum() < 50:
        continue
    
    # Video-level split for this language
    lang_vids = list(set(vids[mask]))
    np.random.seed(42)
    np.random.shuffle(lang_vids)
    n_test = max(1, int(len(lang_vids) * 0.3))
    lang_test = set(lang_vids[:n_test])
    
    test_m = test_mask & mask
    lang_test_m = np.isin(vids, list(lang_test)) & mask
    
    if lang_test_m.sum() < 10:
        continue
    
    y_pred_lang = lr.predict(X[lang_test_m])
    f1 = f1_score(y[lang_test_m], y_pred_lang)
    print(f'  {lang}: F1={f1:.4f} ({lang_test_m.sum()} test segs)')